# 🚀 AutoPilot — Professional AI Video Factory
**Keep PRIVATE** | GPU: T4 x1 required | 100% Free Stack

## Stack
| Stage | Tool | Free? | VRAM |
|---|---|---|---|
| Script | Groq Llama 3.3 70B | ✅ Free | 0 (API) |
| Voice | Chatterbox TTS (MIT) | ✅ Free | ~2 GB |
| Video | Wan2.1 1.3B (Apache 2.0) | ✅ Free | ~8 GB |
| Images | Pollinations FLUX (no key) | ✅ Free | 0 (API) |
| Music | ACE-Step 1.5 (MIT) | ✅ Free | ~4 GB |
| Captions | Word-level ASS (built-in) | ✅ Free | 0 |
| Thumbnail | FLUX + Pillow (no key) | ✅ Free | 0 (API) |
| Upload | YouTube Data API v3 | ✅ Free | 0 |

## Cells
| Cell | What it does | Time |
|---|---|---|
| 1 | System + Python packages | ~3 min (first run) |
| 2 | GPU models (Chatterbox + Wan2.1) | ~5 min (downloads weights once) |
| 3 | API keys | 30 sec |
| 4 | Run pipeline | ~35-45 min per video |
| 5 | View + download results | Instant |
| 6 | Batch mode (overnight) | ~4-8 hrs |

## Kaggle Secrets Setup
Go to **Add-ons → Secrets** and add:
- `GROQ_API_KEYS` — comma-separated (free at console.groq.com)
- `GEMINI_API_KEYS` — from aistudio.google.com (free)
- `PEXELS_API_KEYS` — from pexels.com/api (free)
- `TAVILY_API_KEY` — from app.tavily.com (1000 free/month)
- `YOUTUBE_CLIENT_SECRET_JSON` — from Google Cloud Console (optional, for upload)

In [ ]:
# =============================================================================
# CELL 1 — BULLETPROOF SYSTEM SETUP & SELF-RESTARTING RESOLVER
# =============================================================================
# Run ONCE. Installs packages and auto-restarts the kernel.
# After restart: skip Cell 1, go straight to Cell 2!
# =============================================================================
import subprocess, sys, os

CLONE_DIR    = '/kaggle/working/autopilot'
PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# Pass this to ALL subprocess calls so chatterbox/transformers use eager attention
_SETUP_ENV = {**os.environ, 'TRANSFORMERS_ATTN_IMPLEMENTATION': 'eager'}
# Correct Wan2.1 model ID (Wan-AI org, not old Wan-Video)
os.environ['WAN21_MODEL_ID'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'
_SETUP_ENV['WAN21_MODEL_ID'] = 'Wan-AI/Wan2.1-T2V-1.3B-Diffusers'

# ── Fast health-check: if env is already good, skip the whole setup ──────────
env_ok = False
try:
    import numpy as np
    import scipy.special
    import diffusers
    from diffusers import WanPipeline
    scipy.special.sph_legendre_p(0, 0, 0)
    if np.__version__ == "1.26.4" and hasattr(diffusers, "WanPipeline"):
        env_ok = True
except Exception:
    pass

if env_ok:
    print("=" * 60)
    print("✅ Environment already verified — all checks passed!")
    print("🚀 Proceed directly to Cell 2.")
    print("=" * 60)
else:
    print("🔧 Environment needs setup. Running installer...")

    # ── 1. Clone / Pull 'clean-push' branch ──────────────────────────────────
    print("\n[1/3] Getting latest code from branch 'clean-push'...")
    if os.path.exists(CLONE_DIR):
        subprocess.run(['git', '-C', CLONE_DIR, 'fetch', 'origin', 'clean-push:clean-push'],
                       capture_output=True, env=_SETUP_ENV)
        subprocess.run(['git', '-C', CLONE_DIR, 'checkout', 'clean-push'],
                       capture_output=True, env=_SETUP_ENV)
        r = subprocess.run(['git', '-C', CLONE_DIR, 'pull', '--ff-only'],
                           capture_output=True, text=True, env=_SETUP_ENV)
        print('      ' + (r.stdout.strip() or 'Already up to date'))
    else:
        r = subprocess.run(
            ['git', 'clone', '--depth', '1', '-b', 'clean-push',
             'https://github.com/rajatsarswat2001/autopilot.git', CLONE_DIR],
            capture_output=True, text=True, env=_SETUP_ENV
        )
        if r.returncode == 0:
            print('      ✅ Cloned clean-push OK')
        else:
            print('      ❌ Clone failed: ' + r.stderr[-200:])
            raise RuntimeError("Git clone failed!")

    # ── 2. Run kaggle_setup.py (passes eager env to all pip subprocesses) ────
    print("\n[2/3] Running kaggle_setup.py...")
    setup_path = os.path.join(CLONE_DIR, 'kaggle_setup.py')
    subprocess.run([sys.executable, setup_path],
                   capture_output=False, env=_SETUP_ENV)

    # ── 3. Kernel restart — required to load newly installed binaries ────────
    print("\n[3/3] Restarting kernel to activate new packages...")
    print("🔄 KERNEL RESTARTING — wait ~3 seconds, then run Cell 2 directly.")
    print("=" * 60)
    os._exit(0)


In [ ]:
# =============================================================================
# CELL 2 -- PRE-LOAD GPU MODELS IN SUBPROCESS (Zero-VRAM Leakage Guard)
# =============================================================================
# Downloads and verifies weights in isolated subprocesses.
# This ensures all GPU memory is 100% released before running the main pipeline!
# =============================================================================
import subprocess, sys, os, torch, time

if not torch.cuda.is_available():
    print('❌ No GPU available — pipeline will run in CPU-only mode (Pexels + Edge TTS)')
else:
    num_gpus = torch.cuda.device_count()
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPUs detected: {num_gpus}x {gpu_name} ({vram_gb:.1f} GB VRAM each)')
    if num_gpus >= 2:
        print('  Dual-T4 mode:')
        print('    cuda:0 -> Chatterbox TTS (~2 GB, freed after audio)')
        print('    cuda:0 -> CogVideoX-2B INT8 (mirror strategy, ~6.5 GB)')
        print('    cuda:1 -> CogVideoX-2B INT8 (mirror strategy, ~6.5 GB)')
    print()

    # Make sure torchao and diffusers are fully installed first
    print('Checking dependencies...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchao>=0.4.0', 'diffusers>=0.33.0'], check=False)

    # -- [1/2] Chatterbox TTS (cuda:0) in subprocess --
    print('\n[1/2] Chatterbox TTS -- checking/downloading weights in subprocess (~1.8 GB first time)...')
    t0 = time.time()
    chat_code = """
import os, torch, gc
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
try:
    from chatterbox.tts import ChatterboxTTS
    model = ChatterboxTTS.from_pretrained(device='cuda:0')
    print('      Chatterbox weights pre-loaded successfully!')
except Exception as e:
    print('      FAIL:', str(e)[:150])
    raise e
"""
    r = subprocess.run([sys.executable, '-c', chat_code], capture_output=True, text=True)
    if r.returncode == 0:
        print(r.stdout.strip())
        print(f'  ✅ OK  Chatterbox ready ({time.time()-t0:.0f}s)')
    else:
        print(f'  ⚠️  Chatterbox failed or skipped: ' + r.stdout.strip() + '\n' + r.stderr.strip()[-200:])
        print('      -> Will fall back to Kokoro TTS fallback at runtime')

    # -- [2/2] CogVideoX-2B INT8 (cuda:1) in subprocess --
    video_gpu_id = 1 if num_gpus >= 2 else 0
    print(f'\n[2/2] CogVideoX-2B INT8 -- checking/downloading weights in subprocess (~5 GB first time)...')
    t0 = time.time()
    cog_code = f"""
import os, torch, gc
os.environ['TRANSFORMERS_ATTN_IMPLEMENTATION'] = 'eager'
from diffusers import CogVideoXPipeline
print('      Downloading and loading pipeline...')
pipe = CogVideoXPipeline.from_pretrained(
    'THUDM/CogVideoX-2b',
    torch_dtype=torch.float16,
)
try:
    from torchao.quantization import quantize_, int8_weight_only
    quantize_(pipe.transformer, int8_weight_only())
    print('      INT8 quantization applied (6.5 GB VRAM)')
except Exception as qe:
    print('      INT8 skipped:', str(qe))
pipe.to('cuda:{video_gpu_id}')
for method in ('enable_vae_slicing', 'enable_vae_tiling', 'enable_attention_slicing'):
    if hasattr(pipe, method):
        try: getattr(pipe, method)()
        except Exception: pass
print('      Running 5-step smoke test...')
with torch.inference_mode():
    pipe(
        prompt='a calm ocean wave, cinematic, photorealistic',
        height=480, width=720, num_frames=49, num_inference_steps=5,
        guidance_scale=6.0,
    )
print('      Smoke test completed successfully!')
"""
    r = subprocess.run([sys.executable, '-c', cog_code], capture_output=True, text=True)
    if r.returncode == 0:
        print(r.stdout.strip())
        print(f'  ✅ OK  CogVideoX-2B ready ({time.time()-t0:.0f}s)')
    else: 
        print(f'  ⚠️  CogVideoX failed or skipped: ' + r.stdout.strip() + '\n' + r.stderr.strip()[-200:])
        print('      -> Will fall back to LTX-Video or Pollinations FLUX at runtime')

    print('\nVRAM status inside parent Jupyter kernel: 0 bytes allocated (Subprocess isolated)')
    print('✅ Setup complete -- proceed directly to Cell 3!')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 3 — API KEYS & SETTINGS (Secure Kaggle Secrets + Fallbacks)
# ═══════════════════════════════════════════════════════════════════════════
import os

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── 1. Load from Kaggle Secrets (Recommended & Secure) ─────────────────────
# Go to "Add-ons -> Secrets" in Kaggle's top menu and add your keys there!
keys = {}
try:
    from kaggle_secrets import UserSecretsClient
    client = UserSecretsClient()
    for k in [
        'GROQ_API_KEYS', 'GEMINI_API_KEYS', 'DEEPSEEK_API_KEY',
        'NVIDIA_API_KEY', 'TAVILY_API_KEY',
        'PEXELS_API_KEYS', 'PEXELS_API_KEY',  # try both spellings
        'PIXABAY_KEY', 'HF_TOKEN',
        'YOUTUBE_API_KEY', 'YOUTUBE_CLIENT_SECRET',
    ]:
        try:
            val = client.get_secret(k)
            if val:
                keys[k] = val
        except Exception:
            pass
except Exception:
    pass

# Write loaded secrets to environment
valid_keys = {k: v for k, v in keys.items() if v and len(v) > 4}
for k, v in valid_keys.items():
    os.environ[k] = v
# Also alias singular/plural Pexels key
if 'PEXELS_API_KEYS' in valid_keys and 'PEXELS_API_KEY' not in valid_keys:
    os.environ['PEXELS_API_KEY'] = valid_keys['PEXELS_API_KEYS']
if 'PEXELS_API_KEY' in valid_keys and 'PEXELS_API_KEYS' not in valid_keys:
    os.environ['PEXELS_API_KEYS'] = valid_keys['PEXELS_API_KEY']

print(f'Loaded {len(valid_keys)} secrets from Kaggle Secrets')
for k in sorted(valid_keys):
    print(f'  {k}: {str(valid_keys[k])[:4]}...')

# ── 2. Pipeline settings ─────────────────────────────────────────────────────
import torch
has_gpu = torch.cuda.is_available()

settings = {
    # ── Core ────────────────────────────────────────────────────────────────
    'AUTOPILOT_AUTO_APPROVE': '1',
    'LOG_LEVEL':              'INFO',
    'FORMAT':                 'short',

    # ── TTS ─────────────────────────────────────────────────────────────────
    'AUDIO_PARALLEL_WORKERS': '1',   # always sequential (Chatterbox is GPU)

    # ── Video generation (new video_gen_tools.py module) ─────────────────────
    'VIDEO_GEN_ENABLED':      '1' if has_gpu else '0',
    'VIDEO_GEN_INT8':         '1',   # CogVideoX INT8: 10 GB → 6.5 GB VRAM
    'VIDEO_GEN_STRATEGY':     'mirror' if (has_gpu and torch.cuda.device_count() >= 2) else 'sequential',
    #   mirror     = CogVideoX on BOTH GPUs simultaneously (2x speed)
    #   hybrid     = LTX-Video(cuda:0) + CogVideoX(cuda:1) simultaneously
    #   sequential = one clip at a time (safe for single GPU)
    'VIDEO_GEN_COG_STEPS':    '25',  # 25 steps = good quality (~2-3 min/clip)
    'VIDEO_GEN_LTX_STEPS':    '8',   # 8 steps = distilled fast fallback

    # ── Old Wan2.1 switches (must be OFF — Wan2.1 is no longer used) ─────────
    'WAN21_ENABLED':          '0',   # DISABLED — replaced by video_gen_tools
    'VISUAL_PARALLEL_WORKERS': '1',  # scene-level sequential (video_gen_tools handles parallelism)

    # ── Stock clips ──────────────────────────────────────────────────────────
    'DISABLE_STOCK':          '1',   # 100% AI-generated (required for monetization)
}

env_path = os.path.join(PIPELINE_DIR, '.env')
with open(env_path, 'w') as f:
    for k, v in settings.items():
        f.write(f'{k}={v}\n')
        os.environ[k] = str(v)

n_gpus = torch.cuda.device_count() if has_gpu else 0
print(f'.env written: {env_path}')
print(f'GPU mode: {n_gpus}x T4 — strategy: {settings["VIDEO_GEN_STRATEGY"]}')
print(f'  -> CogVideoX-2B INT8 (primary) + LTX-Video (auto fallback)')
print(f'  -> Wan2.1 DISABLED (no disk limit risk)')
print(f'  -> Stock clips DISABLED (100% AI, monetization-ready)')
print('\n✅ Run Cell 4 to generate a video')


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 4 — RUN PIPELINE
#
# GPU mode (Wan2.1 ON):  ~35-45 min per 60s video on T4
# CPU mode (Pexels only): ~8-12 min per 60s video
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── CONFIG ───────────────────────────────────────────────────────────────────
# personal_finance | saas_tools | legal_tax | senior_health | storytelling
NICHE = 'personal_finance'
TOPIC = ''     # leave empty = auto-detect trending topic via Tavily + pytrends

# ── GPU VRAM check ───────────────────────────────────────────────────────────
import torch
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    used = torch.cuda.memory_allocated(0) / 1e9
    free = vram - used
    print(f'VRAM: {free:.1f} GB free / {vram:.1f} GB total')
    if free < 10:
        print('⚠️  Low VRAM — forcing WAN21_ENABLED=0 (using Pexels instead)')
        os.environ['WAN21_ENABLED'] = '0'
    else:
        print('✅ Sufficient VRAM for Wan2.1 + Chatterbox')
else:
    print('⚠️  No GPU — running in CPU-only mode (Pexels + Edge TTS)')

# ── Pre-run checks ───────────────────────────────────────────────────────────
checks = {
    'Pipeline dir': os.path.exists(PIPELINE_DIR),
    'main.py':      os.path.exists(os.path.join(PIPELINE_DIR, 'main.py')),
    '.env':         os.path.exists(os.path.join(PIPELINE_DIR, '.env')),
}
print('\nPRE-RUN CHECKS')
for k, v in checks.items():
    print(f'  {"✅" if v else "❌"} {k}')
    if not v:
        raise RuntimeError(f'{k} missing — run Cell 1 and Cell 3 first')

print(f'  Niche: {NICHE}')
print(f'  Topic: {TOPIC or "auto-detect"}')
print('=' * 60)

# Set NICHE as env var so visual_director picks it up
os.environ['NICHE'] = NICHE

cmd = [
    sys.executable, 'main.py',
    '--niche', NICHE,
    '--no-db',
    '--approve',
    '--log-format', 'console',
]
if TOPIC:
    cmd += ['--topic', TOPIC]

start = time.time()
proc = subprocess.Popen(
    cmd,
    cwd=PIPELINE_DIR,
    env=os.environ.copy(),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

all_lines = []
for line in proc.stdout:
    print(line, end='', flush=True)
    all_lines.append(line)
proc.wait()

elapsed = time.time() - start
print('=' * 60)
print(f'EXIT CODE : {proc.returncode}')
print(f'DURATION  : {elapsed:.0f}s ({elapsed/60:.1f} min)')

if proc.returncode != 0:
    print('\n❌ FAILED — last 30 lines:')
    print(''.join(all_lines[-30:]))
else:
    print('\n✅ SUCCESS!')
    vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
    videos = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
    print(f'Videos generated: {len(videos)}')
    for v in videos[:5]:
        mb = v.stat().st_size / 1024 / 1024
        thumb = vid_dir / (v.stem + '_thumb.jpg')
        has_thumb = '🖼️' if thumb.exists() else ''
        print(f'  {v.name}  ({mb:.1f} MB) {has_thumb}')
    print('\nRun Cell 5 to view thumbnails and download')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 5 — VIEW & DOWNLOAD RESULTS
# ═══════════════════════════════════════════════════════════════════════════
from pathlib import Path
from IPython.display import HTML, Image, display
import base64

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'
vid_dir  = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos   = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)

if not videos:
    print('No videos yet — run Cell 4 first')
else:
    total_mb = sum(v.stat().st_size for v in videos) / 1024 / 1024
    print(f'Found {len(videos)} videos ({total_mb:.1f} MB total)')
    print('Download from the Output tab (right panel in Kaggle UI)\n')

    rows = []
    for v in videos:
        mb   = v.stat().st_size / 1024 / 1024
        thumb = vid_dir / (v.stem + '_thumb.jpg')
        ass   = vid_dir / (v.stem + '_captions.ass')

        if thumb.exists():
            img_data = base64.b64encode(thumb.read_bytes()).decode()
            thumb_html = f'<img src="data:image/jpeg;base64,{img_data}" width="240" style="border-radius:8px">'
        else:
            thumb_html = '<div style="width:240px;height:135px;background:#333;border-radius:8px;display:flex;align-items:center;justify-content:center;color:#888">No thumbnail</div>'

        caption_badge = '✅ Captions' if ass.exists() else ''
        rows.append(
            f'<tr style="border-bottom:1px solid #333">'
            f'<td style="padding:10px">{thumb_html}</td>'
            f'<td style="padding:10px;vertical-align:top">'
            f'<b style="font-size:14px">{v.name}</b><br>'
            f'<span style="color:#aaa">{mb:.1f} MB</span><br>'
            f'<span style="color:#4CAF50">{caption_badge}</span>'
            f'</td></tr>'
        )

    html = (
        '<div style="background:#1a1a1a;padding:20px;border-radius:12px">'
        '<h2 style="color:#fff;margin-top:0">🎬 Generated Videos</h2>'
        '<table style="border-collapse:collapse;width:100%">'
        + ''.join(rows)
        + '</table></div>'
    )
    display(HTML(html))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CELL 6 — BATCH MODE  (multiple niches, run overnight)
#
# T4 GPU (30 hrs/week quota) can produce ~40-45 videos/week
# GPU mode: ~40 min/video → 6 hr = ~9 videos overnight
# CPU mode: ~10 min/video → 6 hr = ~36 videos overnight (Pexels only)
# ═══════════════════════════════════════════════════════════════════════════
import subprocess, sys, os, time
from pathlib import Path

PIPELINE_DIR = '/kaggle/working/autopilot/autopilot_pipeline'

# ── CONFIG ───────────────────────────────────────────────────────────────────
# Each entry: (niche, topic_override_or_None)
BATCH_JOBS = [
    ('personal_finance',  None),   # auto-detect trending topic
    ('saas_tools',        None),
    ('personal_finance',  None),   # second video, different trending topic
    # ('legal_tax',       None),
    # ('senior_health',   None),
    # ('storytelling',    'The untold story of the Manhattan Project'),
]

import torch
gpu_mode = torch.cuda.is_available()
est_min  = 40 if gpu_mode else 10
print(f'Batch: {len(BATCH_JOBS)} videos')
print(f'Mode: {"GPU (Wan2.1 + Chatterbox)" if gpu_mode else "CPU (Pexels + Edge TTS)"}')
print(f'Estimated time: ~{len(BATCH_JOBS) * est_min} min ({len(BATCH_JOBS) * est_min / 60:.1f} hrs)')
print('=' * 60)

results = []
total_start = time.time()

for i, (niche, topic) in enumerate(BATCH_JOBS, 1):
    print(f'\n[{i}/{len(BATCH_JOBS)}] niche={niche} topic={topic or "auto"} ...')
    os.environ['NICHE'] = niche
    t0 = time.time()

    cmd = [
        sys.executable, 'main.py',
        '--niche', niche,
        '--no-db', '--approve',
        '--log-format', 'console'
    ]
    if topic:
        cmd += ['--topic', topic]

    r = subprocess.run(
        cmd,
        cwd=PIPELINE_DIR,
        env=os.environ.copy(),
        timeout=7200,  # 2hr max per video
        capture_output=True, text=True
    )
    elapsed = time.time() - t0
    ok = r.returncode == 0
    results.append((niche, topic or 'auto', ok, elapsed))
    status = '✅' if ok else '❌'
    print(f'  {status} ({elapsed:.0f}s / {elapsed/60:.1f} min)')
    if not ok:
        lines = (r.stdout + r.stderr).split('\n')
        print('\n'.join(lines[-10:]))

    # Clear GPU between videos
    if gpu_mode:
        import gc
        torch.cuda.empty_cache()
        gc.collect()

total_elapsed = time.time() - total_start
print(f'\n{"="*60}')
print(f'BATCH COMPLETE  ({total_elapsed/60:.1f} min total)')
success = sum(1 for _, _, ok, _ in results if ok)
print(f'Results: {success}/{len(results)} succeeded')
for niche, topic, ok, t in results:
    print(f'  {"✅" if ok else "❌"}  {niche:<25}  {topic:<20}  {t/60:.1f} min')

vid_dir = Path(PIPELINE_DIR) / 'outputs' / 'video'
videos  = sorted(vid_dir.glob('*.mp4'), key=lambda x: x.stat().st_mtime, reverse=True)
print(f'\nTotal videos in output: {len(videos)}')
for v in videos:
    print(f'  {v.name}  ({v.stat().st_size/1024/1024:.1f} MB)')